<a href="https://colab.research.google.com/github/raghavxi1/gym-attendance-analysis/blob/main/notebooks/01_data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd, numpy as np
from google.colab import files
files.upload()

Saving daily_gym_attendance_workout_data.csv to daily_gym_attendance_workout_data.csv


{'daily_gym_attendance_workout_data.csv': b'member_id,visit_date,age,gender,membership_type,workout_type,workout_duration_minutes,calories_burned,check_in_time,attendance_status\n1,2024-10-11,64,Other,Annual,HIIT,28,171,20:04,Absent\n2,2024-06-01,65,Female,Quarterly,Strength Training,72,650,19:17,Absent\n3,2024-06-13,45,Male,Quarterly,Cardio,70,633,7:24,Absent\n4,2024-02-05,35,Female,Monthly,CrossFit,64,362,7:18,Absent\n5,2024-07-13,26,Female,Quarterly,Yoga,31,262,11:22,Absent\n6,2024-09-11,45,Other,Quarterly,Cardio,63,518,20:08,Present\n7,2024-02-22,30,Other,Quarterly,CrossFit,50,404,18:17,Absent\n8,2024-05-04,38,Male,Annual,Yoga,99,959,21:59,Absent\n9,2024-02-10,61,Female,Monthly,CrossFit,36,289,9:39,Present\n10,2024-08-14,36,Other,Quarterly,Strength Training,94,643,11:29,Absent\n11,2024-10-11,60,Female,Monthly,Strength Training,110,738,22:43,Present\n12,2024-12-25,26,Female,Monthly,HIIT,86,547,8:17,Absent\n13,2024-06-28,48,Female,Quarterly,HIIT,44,389,19:08,Absent\n14,2024-12-01,54,

In [2]:
df=pd.read_csv('daily_gym_attendance_workout_data.csv')
df.head()

,member_id,visit_date,age,gender,membership_type,workout_type,workout_duration_minutes,calories_burned,check_in_time,attendance_status
0,1,2024-10-11,64,Other,Annual,HIIT,28,171,20:04,Absent
1,2,2024-06-01,65,Female,Quarterly,Strength Training,72,650,19:17,Absent
2,3,2024-06-13,45,Male,Quarterly,Cardio,70,633,7:24,Absent
3,4,2024-02-05,35,Female,Monthly,CrossFit,64,362,7:18,Absent
4,5,2024-07-13,26,Female,Quarterly,Yoga,31,262,11:22,Absent


In [3]:
df.info()
print("nulls:\n", df.isna().sum())
print("Duplicate rows:", df.duplicated().sum())
print("Unique members:", df['member_id'].nunique(), "| Total rows:", len(df))
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2600 entries, 0 to 2599
Data columns (total 10 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   member_id                 2600 non-null   int64 
 1   visit_date                2600 non-null   object
 2   age                       2600 non-null   int64 
 3   gender                    2600 non-null   object
 4   membership_type           2600 non-null   object
 5   workout_type              2600 non-null   object
 6   workout_duration_minutes  2600 non-null   int64 
 7   calories_burned           2600 non-null   int64 
 8   check_in_time             2600 non-null   object
 9   attendance_status         2600 non-null   object
dtypes: int64(4), object(6)
memory usage: 203.3+ KB
nulls:
 member_id                   0
visit_date                  0
age                         0
gender                      0
membership_type             0
workout_type                0
worko

,member_id,age,workout_duration_minutes,calories_burned
count,2600.000000,2600.000000,2600.000000,2600.000000
mean,1300.500000,41.947308,70.162692,523.775385
std,750.699674,13.862067,28.610264,240.449717
min,1.000000,18.000000,20.000000,102.000000
25%,650.750000,30.000000,46.000000,329.000000
50%,1300.500000,42.000000,70.000000,504.000000
75%,1950.250000,54.000000,94.000000,691.250000
max,2600.000000,65.000000,120.000000,1186.000000


In [4]:
for c in ['gender','membership_type','workout_type','attendance_status']:
    print(df[c].value_counts(dropna=False), '\n')

gender
Female    886
Other     863
Male      851
Name: count, dtype: int64 

membership_type
Annual       903
Monthly      856
Quarterly    841
Name: count, dtype: int64 

workout_type
Strength Training    559
Cardio               545
HIIT                 520
CrossFit             509
Yoga                 467
Name: count, dtype: int64 

attendance_status
Absent     1350
Present    1250
Name: count, dtype: int64 



In [12]:
df = pd.read_csv('daily_gym_attendance_workout_data.csv')

df.columns = df.columns.str.strip().str.lower()
df = df.drop_duplicates().copy()

df['visit_date'] = pd.to_datetime(df['visit_date'])
t = pd.to_datetime(df['check_in_time'].astype(str).str.strip(), format='%H:%M', errors='coerce')
assert t.notna().all(), "Some times failed to parse"

df['check_in_hour'] = t.dt.hour
df['check_in_time'] = t.dt.strftime('%H:%M')
df['month'] = df['visit_date'].dt.month
df['month_name'] = df['visit_date'].dt.month_name()
df['weekday'] = df['visit_date'].dt.day_name()
df['is_weekend'] = df['visit_date'].dt.dayofweek >= 5
df['age_group'] = pd.cut(df['age'], [0,24,34,44,54,120],
                         labels=['<25','25-34','35-44','45-54','55+'])
df['is_present'] = (df['attendance_status'] == 'Present').astype(int)

counts = df['check_in_hour'].value_counts().sort_index()
print(len(counts), counts.sum())   # expect 18 2600

from scipy.stats import chisquare
print(chisquare(counts))

18 2600
Power_divergenceResult(statistic=np.float64(27.072307692307692), pvalue=np.float64(0.05701584883058561))


In [13]:
df.to_csv('gym_clean.csv', index=False)
from google.colab import files
files.download('gym_clean.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>